# Victim detection: train on Colab

Runtime → Change runtime type → **T4 GPU** (or better). Run top to bottom.
Trains YOLO11n per modality: thermal on HIT-UAV, RGB on C2A. Evaluates on each test split and on the WiSARD real-SAR sample.
Details: `src/detection/README.md`, `research/datasets/detection-data-analysis.md`.

In [ ]:
# 1. Code + deps. Private repo: use https://<TOKEN>@github.com/sagyzdop/capstone.git
!git clone -q https://github.com/sagyzdop/capstone.git
%cd capstone
!pip -q install ultralytics gdown

In [ ]:
# 2. Data (≈ 6.7 GB total). Comment out what you don't need.
!mkdir -p data/raw/hit-uav data/raw/c2a data/raw/wisard
# HIT-UAV thermal, 815 MB, CC0
!curl -sL -o data/raw/hit-uav/hit-uav.zip "https://zenodo.org/api/records/7633134/files/suojiashun/HIT-UAV-Infrared-Thermal-Dataset-v1.2.1.zip/content"
!cd data/raw/hit-uav && md5sum hit-uav.zip && unzip -q hit-uav.zip -d extracted   # md5 bb6c5a33659b7507eae83458729c6109
# C2A RGB, 4.9 GB
!gdown -q 1Uba6CHJRCvF-rgfCXDR6NhTnOvngWsfe -O data/raw/c2a/c2a.zip
!python -c "import zipfile; zipfile.ZipFile('data/raw/c2a/c2a.zip').extractall('data/raw/c2a/extracted')"
# WiSARD multi-modal sample, 1 GB (eval only)
!gdown -q 1uSgMXuZGVCrWM_151UcykyHejxxaVHOo -O data/raw/wisard/wisard-sample.zip
!cd data/raw/wisard && unzip -q wisard-sample.zip -d extracted

In [ ]:
# 3. Convert to single-class `person` YOLO datasets -> data/yolo/*.yaml
R = "data/raw/hit-uav/extracted/suojiashun-HIT-UAV-Infrared-Thermal-Dataset-b53106c/normal_json"
!python src/detection/prepare.py hit-uav --person-ids 0 --split train={R}/train:{R}/annotations/train.json --split val={R}/val:{R}/annotations/val.json --split test={R}/test:{R}/annotations/test.json
C = "data/raw/c2a/extracted/new_dataset3"
!python src/detection/prepare.py c2a --person-ids 0 --split train={C}/train/images:{C}/train/labels --split val={C}/val/images:{C}/val/labels --split test={C}/test/images:{C}/test/labels
import glob
for m in ["VIS", "IR"]:
    d = glob.glob(f"data/raw/wisard/extracted/*_{m}_*")[0]
    !python src/detection/prepare.py wisard-{m.lower()} --person-ids 0 --split val={d}:{d}

In [ ]:
# 4. Train. Thermal first (~2.9k imgs, fast), then RGB (~10k imgs).
# Add more runs (imgsz=960, model=yolo11s.pt) as separate lines if time allows.
!yolo detect train model=yolo11n.pt data=data/yolo/hit-uav.yaml epochs=50 patience=15 imgsz=640 batch=32 project=runs name=hituav-n-640 exist_ok=True
!yolo detect train model=yolo11n.pt data=data/yolo/c2a.yaml epochs=50 patience=10 imgsz=640 batch=32 project=runs name=c2a-n-640 exist_ok=True

In [ ]:
# 5. Evaluate: own test split + real-SAR WiSARD sample (thermal model on IR, RGB model on VIS)
import glob
w = lambda name: glob.glob(f"runs/**/{name}/weights/best.pt", recursive=True)[0]
!yolo detect val model={w('hituav-n-640')} data=data/yolo/hit-uav.yaml split=test imgsz=640 name=eval-hituav-test
!yolo detect val model={w('hituav-n-640')} data=data/yolo/wisard-ir.yaml split=val imgsz=640 name=eval-hituav-on-wisard-ir
!yolo detect val model={w('c2a-n-640')} data=data/yolo/c2a.yaml split=test imgsz=640 name=eval-c2a-test
!yolo detect val model={w('c2a-n-640')} data=data/yolo/wisard-vis.yaml split=val imgsz=640 name=eval-c2a-on-wisard-vis
!yolo detect val model={w('c2a-n-640')} data=data/yolo/wisard-vis.yaml split=val imgsz=1280 name=eval-c2a-on-wisard-vis-1280

In [ ]:
# 6. Save weights + metrics to Drive (or download the zip from the Files panel)
!zip -qr results.zip runs -i "*best.pt" "*results.csv" "*.png"
from google.colab import files
files.download("results.zip")